# Démo : Injection par un outil MCP

**Contexte :** assistant du service client de Lagune Banque.

La description d’un outil fourni par un serveur tiers peut contenir des instructions malveillantes. Cette démonstration utilise un faux convertisseur de devises local pour observer leur effet sur un agent et présenter les contrôles côté hôte.

**Sommaire**
- [Configuration](#configuration)
- [Description exposée au modèle](#description-exposee-au-modele)
- [Agent sans protection](#agent-sans-protection)
- [Contrôles côté hôte](#controles-cote-hote)


## <a id="configuration"></a>Configuration

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path("."), Path(".."), Path("../.."), Path("../../.."), Path("starter"), Path("../../starter")] if (p / "code_partage").exists()).resolve()
sys.path[:0] = [str(RACINE / "code_partage"), str(RACINE / "04_mcp_a2a" / "socle")]

from config import prepare
from outils import AUJOURDHUI, KOUASSI, print_trace, new_model, reset_db
from pont_mcp import MCPConnection, stdio_transport

In [ ]:
prepare()
reset_db()

In [ ]:
PROMPT = f"""Tu es l'assistant automatisé du service client de Lagune Banque. Nous sommes le {AUJOURDHUI}.
Tu réponds en français, en phrases courtes. Tu consultes les prêts et les échéances avec les outils disponibles et tu
n'inventes jamais une date ou un montant. Tu ne prends aucune décision de crédit."""

In [ ]:
import tempfile

DOSSIER = next(p for p in [Path("04_mcp_a2a/demos"), Path("."), Path(".."), Path("../../starter/04_mcp_a2a/demos")]
               if (p / "demo1_fastmcp" / "serveur_demo.py").exists()).resolve()
PRETS = stdio_transport(DOSSIER / "demo1_fastmcp" / "serveur_demo.py")
TIERS = stdio_transport(DOSSIER / "demo2_injection" / "serveur_tiers.py")
JOURNAL = Path(tempfile.gettempdir()) / "serveur_tiers_recu.jsonl"   # ce que le serveur tiers reçoit

## <a id="description-exposee-au-modele"></a>Description exposée au modèle

Un humain voit « convertisseur de devises ». Le modèle reçoit la description complète, affichée ci-dessous.

In [ ]:
async with MCPConnection(TIERS) as tiers:
    for d in await tiers.definitions():
        print(d.name, "\n", d.description, "\n")
        print("paramètres :", list(d.inputSchema["properties"]))

## <a id="agent-sans-protection"></a>Agent sans protection

L'agent a deux serveurs : les prêts (de confiance) et le convertisseur (tiers). La question posée est anodine.

In [ ]:
from langchain.agents import create_agent

QUESTION = "Je suis le client CLI-0002. Combien font 15 000 XOF en euros ?"

async def lancer(outils_tiers_fn):
    """outils_tiers_fn(tiers) renvoie les outils du serveur tiers que l'on accepte de donner à l'agent."""
    JOURNAL.unlink(missing_ok=True)
    async with MCPConnection(PRETS) as prets, MCPConnection(TIERS) as tiers:
        outils = await prets.tools() + await outils_tiers_fn(tiers)
        agent = create_agent(new_model(), outils, system_prompt=PROMPT + "\nLe client identifié est CLI-0002.")
        resultat = await agent.ainvoke({"messages": [{"role": "user", "content": QUESTION}]})
    print_trace(resultat)
    print("\n--- Reçu par le serveur tiers ---")
    print(JOURNAL.read_text(encoding="utf-8") if JOURNAL.exists() else "(rien)")

await lancer(lambda tiers: tiers.tools())

Si le modèle obéit à la description, le serveur tiers reçoit les prêts et les échéances du client, et la réponse affichée ne parle que de conversion : le client ne voit rien. Le modèle peut aussi résister, selon le modèle et l'essai. La protection ne doit donc pas dépendre du modèle.

## <a id="controles-cote-hote"></a>Contrôles côté hôte

L'hôte ne fait pas confiance à la description d'un serveur tiers :

1. Réécrire la définition : l'hôte fournit sa propre description et ne garde que les paramètres attendus (`montant`, `de`, `vers`). Le paramètre `contexte` disparaît pour le modèle.
2. Appliquer le moindre privilège : un serveur tiers reçoit uniquement les outils nécessaires, et un outil qui touche aux données bancaires reste en lecture seule.
3. Relire les descriptions à l'installation d'un serveur et à chaque mise à jour.

```mermaid
flowchart LR
    T["Serveur tiers : convertir_devise"] -->|"description et paramètres"| H["Hôte : revue et réécriture"]
    P["Serveur prêts de confiance"] -->|"outils"| H
    H -->|"description de l'hôte, 3 paramètres"| M["Modèle"]
    M -->|"appel de convertir_devise"| H
    H -->|"montant, de, vers"| T
```

In [ ]:
import re
from langchain_core.tools import StructuredTool

MOTIFS_SUSPECTS = [r"<important>", r"ne mentionne jamais", r"ne dis pas", r"ignore", r"avant de répondre", r"obtenu avec les autres outils"]

def description_suspecte(texte):
    return [m for m in MOTIFS_SUSPECTS if re.search(m, texte, re.I)]

async def outils_tiers_sur(tiers):
    sortie = []
    for d in await tiers.definitions():
        print(f"revue de {d.name} : motifs suspects = {description_suspecte(d.description) or 'aucun'}")
        if d.name != "convertir_devise":
            continue                                    # moindre privilège : on n'expose que ce dont on a besoin

        async def convertir(montant: float, de: str, vers: str) -> str:
            r = await tiers.client.call_tool("convertir_devise", {"montant": montant, "de": de, "vers": vers})
            return r.content[0].text

        sortie.append(StructuredTool.from_function(
            coroutine=convertir, name="convertir_devise",
            description="Convertit un montant d'une devise vers une autre (XOF, EUR, USD).",   # description écrite par NOUS
            args_schema={"type": "object", "properties": {"montant": {"type": "number"}, "de": {"type": "string"},
                                                          "vers": {"type": "string"}}, "required": ["montant", "de", "vers"]}))
    return sortie

await lancer(outils_tiers_sur)

Le serveur tiers ne reçoit plus que `montant`, `de` et `vers`. Une description d'outil est une consigne que le modèle exécute. Un serveur MCP tiers hérite des permissions que vous lui accordez : exigez une authentification et un accès en lecture seule par défaut.